In [1]:
from ome_zarr.image import NgffImage, NgffMultiscales
from ome_zarr.scene import NgffScene
from ome_zarr_models._v06.coordinate_transforms import CoordinateSystem, CoordinateSystemIdentifier, Axis, Translation
from bioio import BioImage
import os
import glob
import numpy as np
from pathlib import Path
import dask.array as da
from dask.distributed import Client, get_client
import napari
import tqdm
import xmltodict

from multiview_stitcher import spatial_image_utils as si_utils
from multiview_stitcher import fusion
from skimage import io

In [2]:
root = r'C:\Users\johan\Desktop\local_data\Lightsheet\2026_03_20_11_49_15--Marina2_96\TileScan 2'

In [3]:
try:
  client = get_client()
except ValueError:
  client = Client()

print(client.dashboard_link)

http://127.0.0.1:8787/status


In [4]:
viewer = napari.Viewer()

In [5]:
def find_in_nested_dict(data, key, value):
    """
    Recursively search for objects containing a specific key-value pair.
    Returns a list of all matching objects.
    """
    results = []
    
    def search(obj):
        if isinstance(obj, dict):
            if obj.get(key) == value:
                results.append(obj)
            for v in obj.values():
                search(v)
        elif isinstance(obj, list):
            for item in obj:
                search(item)
    
    search(data)
    return results  

## With acquisitions in row subfolders

In [ ]:
columns = os.listdir(root)

for column in columns:
	column_path = Path(root) / column
	rows = [column_path / d for d in os.listdir(column_path) if os.path.isdir(column_path / d)]

	for row in rows:
		image_files = glob.glob(os.path.join(row, '**', '*.tif'), recursive=True)

		images = []
		transforms = []

		for filename in tqdm.tqdm(image_files):

			basename = Path(filename).stem.split('.')[0]
			outfile_oz = os.path.join(root, basename + ".ome.zarr")
			bf = BioImage(filename)

			if bf.physical_pixel_sizes.Z is None:
				z_scale = 1
			else:
				z_scale = bf.physical_pixel_sizes.Z *1e6

			scale = {
				"z": z_scale,
				"y": bf.physical_pixel_sizes.Y *1e6,
				"x": bf.physical_pixel_sizes.X *1e6
			}

			axes_units = {
				"z": "micrometer",
				"y": "micrometer",
				"x": "micrometer",
			}
			
			# parse tile metadata
			metadata_file = os.path.join(Path(filename).parent, "Metadata", f"{basename}.xlif")
			metadata_dict = xmltodict.parse(open(metadata_file).read())
			tile_scan_info = find_in_nested_dict(metadata_dict, '@Name', 'TileScanInfo')[0]["Tile"]

			# n_series= bf.series_count()
			n_series = len(bf.scenes)

			if n_series > 1:
				sims = []
				for idx in range(n_series):
					bf.set_scene(bf.scenes[idx])
					dask_image = bf.dask_data
					# dask_image = bf.to_dask(series=idx)
					ts_info = tile_scan_info[idx]

					t_z = float(ts_info["@PosZ"]) * 1e6
					t_y = float(ts_info["@PosY"]) * 1e6
					t_x = float(ts_info["@PosX"]) * 1e6

					sims.append(
						si_utils.get_sim_from_array(
							dask_image,
							dims=["t", "c", "z", "y", "x"],
							scale=scale,
							translation={"z": t_z, "y": t_y, "x": t_x},
							transform_key="stage_metadata",
							)
					)

				image = fusion.fuse(sims, transform_key="stage_metadata", fusion_func=fusion.max_fusion).data
			else:
				# image = bf.to_dask(series=0)
				image = bf.dask_data
				t_z = float(tile_scan_info["@PosZ"]) * 1e6
				t_y = float(tile_scan_info["@PosY"]) * 1e6
				t_x = float(tile_scan_info["@PosX"]) * 1e6

			# coerce to zyx
			image = image.squeeze()
			while len(image.shape) < 3:
				image = image[None, ...]

			ngff_image = NgffImage(
				image,
				dims=["z", "y", "x"],
				scale=scale,
				axes_units=axes_units,
				name=basename
			)
			ngff_multiscales = NgffMultiscales(
				image=ngff_image,
				scale_factors=[
				{"z": 2, "y": 2, "x": 2},
				{"z": 4, "y": 4, "x": 4},
				{"z": 8, "y": 8, "x": 8},
				{"z": 16, "y": 16, "x": 16},
				],
			)
			images.append(ngff_multiscales)

			transform = Translation(
				translation=(t_z, t_y, t_x),
				input=CoordinateSystemIdentifier(
					path=ngff_multiscales.name,
					name=ngff_multiscales.metadata.coordinateSystems[0].name
				),
				output=CoordinateSystemIdentifier(
					name="world"
				)
			)
			transforms.append(transform)

		# write a scene
		cs_world = CoordinateSystem(
			name="world",
			axes=(
				Axis(name="z", type="space", unit="micrometer"),
				Axis(name="y", type="space", unit="micrometer"),
				Axis(name="x", type="space", unit="micrometer"),
			)
		)

		ngff_scene = NgffScene(
			images=images,
			transformations=transforms,
			coordinate_systems=(
				cs_world,
			)
		)

		ngff_scene.to_ome_zarr(
			os.path.join(root, column, row.stem, "scene.ome.zarr")
		)
		print(f"Done with {row.stem}")

## With acquisitions in ome-tifs

In [7]:
columns = os.listdir(root)

for column in columns:
	column_path = Path(root) / column
	image_files = glob.glob(os.path.join(column_path, '**', '*.tif'), recursive=True)

	images = []
	transforms = []

	for filename in tqdm.tqdm(image_files):

		basename = Path(filename).stem.split('.')[0]
		outfile_oz = os.path.join(root, basename + ".ome.zarr")
		bf = BioImage(filename)

		if bf.physical_pixel_sizes.Z is None:
			z_scale = 1
		else:
			z_scale = bf.physical_pixel_sizes.Z *1e6

		scale = {
			"z": z_scale,
			"y": bf.physical_pixel_sizes.Y *1e6,
			"x": bf.physical_pixel_sizes.X *1e6
		}

		axes_units = {
			"z": "micrometer",
			"y": "micrometer",
			"x": "micrometer",
		}
		
		# parse tile metadata
		metadata_file = os.path.join(Path(filename).parent, "Metadata", f"{basename}.xlif")
		metadata_dict = xmltodict.parse(open(metadata_file).read())
		tile_scan_info = find_in_nested_dict(metadata_dict, '@Name', 'TileScanInfo')[0]["Tile"]

		# n_series= bf.series_count()
		n_series = len(bf.scenes)

		if n_series > 1:
			sims = []
			for idx in range(n_series):
				bf.set_scene(bf.scenes[idx])
				dask_image = bf.dask_data
				# dask_image = bf.to_dask(series=idx)
				ts_info = tile_scan_info[idx]

				t_z = float(ts_info["@PosZ"]) * 1e6
				t_y = float(ts_info["@PosY"]) * 1e6
				t_x = float(ts_info["@PosX"]) * 1e6

				sims.append(
					si_utils.get_sim_from_array(
						dask_image,
						dims=["t", "c", "z", "y", "x"],
						scale=scale,
						translation={"z": t_z, "y": t_y, "x": t_x},
						transform_key="stage_metadata",
						)
				)

			image = fusion.fuse(sims, transform_key="stage_metadata", fusion_func=fusion.max_fusion).data
		else:
			# image = bf.to_dask(series=0)
			image = bf.dask_data
			t_z = float(tile_scan_info["@PosZ"]) * 1e6
			t_y = float(tile_scan_info["@PosY"]) * 1e6
			t_x = float(tile_scan_info["@PosX"]) * 1e6

		# coerce to zyx
		image = image.squeeze()
		while len(image.shape) < 3:
			image = image[None, ...]

		ngff_image = NgffImage(
			image,
			dims=["z", "y", "x"],
			scale=scale,
			axes_units=axes_units,
			name=basename
		)
		ngff_multiscales = NgffMultiscales(
			image=ngff_image,
			scale_factors=[
			{"z": 2, "y": 2, "x": 2},
			{"z": 4, "y": 4, "x": 4},
			{"z": 8, "y": 8, "x": 8},
			{"z": 16, "y": 16, "x": 16},
			],
		)
		images.append(ngff_multiscales)

		transform = Translation(
			translation=(t_z, t_y, t_x),
			input=CoordinateSystemIdentifier(
				path=ngff_multiscales.name,
				name=ngff_multiscales.metadata.coordinateSystems[0].name
			),
			output=CoordinateSystemIdentifier(
				name="world"
			)
		)
		transforms.append(transform)

	# write a scene
	cs_world = CoordinateSystem(
		name="world",
		axes=(
			Axis(name="z", type="space", unit="micrometer"),
			Axis(name="y", type="space", unit="micrometer"),
			Axis(name="x", type="space", unit="micrometer"),
		)
	)

	ngff_scene = NgffScene(
		images=images,
		transformations=transforms,
		coordinate_systems=(
			cs_world,
		)
	)

	ngff_scene.to_ome_zarr(
		os.path.join(root, column, f"{basename}.ome.zarr")
	)
	print(f"Done with {basename}")

  0%|          | 0/12 [00:00<?, ?it/s]<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
  8%|▊         | 1/12 [00:19<03:29, 19.04s/it]<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
 17%|█▋        | 2/12 [00:36<03:03, 18.33s/it]<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
 25%|██▌       | 3/12 [00:55<02:47, 18.60s/it]<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarni

Done with 9


  0%|          | 0/12 [00:00<?, ?it/s]<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
  8%|▊         | 1/12 [00:25<04:35, 25.05s/it]<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
 17%|█▋        | 2/12 [00:43<03:30, 21.03s/it]<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
 25%|██▌       | 3/12 [01:01<02:56, 19.62s/it]<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarni

Done with 9


  0%|          | 0/12 [00:00<?, ?it/s]<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
  8%|▊         | 1/12 [00:18<03:22, 18.44s/it]<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
 17%|█▋        | 2/12 [00:36<03:03, 18.34s/it]<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
 25%|██▌       | 3/12 [00:54<02:44, 18.29s/it]<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarni

...

Done with 9


  0%|          | 0/12 [00:00<?, ?it/s]<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
  8%|▊         | 1/12 [00:17<03:15, 17.79s/it]<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
 17%|█▋        | 2/12 [00:42<03:37, 21.79s/it]<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
 25%|██▌       | 3/12 [00:59<02:57, 19.75s/it]<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarni

Done with 9


0it [00:00, ?it/s]


Done with 9


In [18]:
os.path.join(root, column, row.stem, "scene.ome.zarr")

'C:\\Users\\johan\\Desktop\\local_data\\BiaPoL\\autoSCAPE\\20260317\\10x\\10\\10\\scene.ome.zarr'

In [ ]:
viewer.open(os.path.join(root, column, "scene.ome.zarr"), plugin="napari-ome-zarr")

{'@FieldX': '0',
 '@FieldY': '0',
 '@PosX': '0.0455994167',
 '@PosY': '0.0479130289',
 '@PosZ': '0.0013600117'}

## Show scenes



In [9]:
for column in columns:
	viewer.open(os.path.join(root, column, "scene.ome.zarr"), plugin="napari-ome-zarr")

Root group {'ome': {'scene': {'coordinateTransformations': [{'type': 'translation', 'input': {'name': 'physical', 'path': 'P1'}, 'output': {'name': 'world'}, 'translation': [1360.0117, 46188.4402, 48583.5365]}, {'type': 'translation', 'input': {'name': 'physical', 'path': 'P2'}, 'output': {'name': 'world'}, 'translation': [1360.0117, 46258.5748, 50074.0541]}, {'type': 'translation', 'input': {'name': 'physical', 'path': 'P3'}, 'output': {'name': 'world'}, 'translation': [1360.0117, 47813.0421, 48576.574499999995]}, {'type': 'translation', 'input': {'name': 'physical', 'path': 'P4'}, 'output': {'name': 'world'}, 'translation': [1360.0117, 47913.028900000005, 50099.4176]}], 'coordinateSystems': [{'name': 'world', 'axes': [{'name': 'z', 'type': 'space', 'unit': 'micrometer'}, {'name': 'y', 'type': 'space', 'unit': 'micrometer'}, {'name': 'x', 'type': 'space', 'unit': 'micrometer'}]}]}, 'version': '0.6'}}
Scene.matches {'scene': {'coordinateTransformations': [{'type': 'translation', 'input

ValueError: Reader 'napari-ome-zarr' was selected to open ['C:\\Users\\johan\\Desktop\\local_data\\BiaPoL\\autoSCAPE\\20260317\\10x\\J\\Metadata\\scene.ome.zarr'], but returned no data.

In [18]:
layer = viewer.layers['scene.ome [1]']
max_projection = layer.data[0].min(axis=0)
viewer.add_image(max_projection, scale = layer.affine.scale[1:], translate=layer.affine.translate[1:], name="max projection")

<Image layer 'max projection [1]' at 0x14656604890>

In [61]:
for layer in viewer.layers:
	projection = layer.data[0].min(axis=0).rechunk(640, 640)

	# iterate over chunks and save as individual images
	for idx in np.ndindex(projection.numblocks):
		chunk = projection.blocks[idx]
		if not chunk.shape == (640, 640):
			continue

		io.imsave(os.path.join(r"C:\Users\johan\Desktop\local_data\BiaPoL\autoSCAPE\20260317\projections", f"{layer.name}_chunk_{idx}.tif"), chunk.compute(), check_contrast=False)

In [46]:
viewer_projections.layers.clear()
viewer_projections.add_image(chunk, contrast_limits=(p5.compute(), p95.compute()))

<Image layer 'chunk' at 0x1d8aa80ea20>

In [ ]:
viewer.layers[0].data[0].min(axis=0).rechunk(640, 640)

(4, 5)

In [55]:
for idx in np.ndindex(projection.numblocks):
  print(idx)
  break

(0, 0)


In [56]:
projection.blocks[idx]

dask.array<blocks, shape=(640, 640), dtype=uint16, chunksize=(640, 640), chunktype=numpy.ndarray>